In [16]:
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import classification_report
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, RobustScaler


In [12]:
df = sns.load_dataset("titanic")

leaky = ["alive", "who", "adult_male", "class", "deck", "embark_town"]      
df = df.drop(columns=leaky)

In [13]:
X = df.drop(columns=["survived"])
y = df["survived"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [14]:
num_cols = ["age", "sibsp", "parch", "fare"]
cat_cols = ["sex", "embarked"]
ord_cols = ["pclass"]

pipe_so  = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  RobustScaler()),
])
pipe_cat = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(drop="first", handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", pipe_so,  num_cols),
    ("cat", pipe_cat, cat_cols),
    ("ord", "passthrough", ord_cols),
])

preprocess.fit(X_train)               # fit CHỈ trên train
X_train_t = preprocess.transform(X_train)
X_test_t = preprocess.transform(X_test)

In [15]:
model = LinearRegression()

model.fit(X_train_t, y_train)
y_pred = model.predict(X_test_t)
y_pred = [1 if y_p > 0.5 else 0 for y_p in y_pred]
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.82      0.88      0.85       110
           1       0.79      0.70      0.74        69

    accuracy                           0.81       179
   macro avg       0.80      0.79      0.79       179
weighted avg       0.81      0.81      0.81       179



## Nhận xét về Linear Regression và Logistic Regression

Trong bài toán phân loại hạt đậu (Dry Bean Classification), **Logistic Regression** cho kết quả tốt hơn **Linear Regression**.

- **Logistic Regression** được thiết kế riêng cho bài toán phân loại. 

- **Linear Regression** vốn là mô hình dành cho bài toán hồi quy, có đầu ra là giá trị liên tục. Để áp dụng cho phân loại, cần chuyển đổi kết quả dự đoán thành nhãn bằng cách đặt một ngưỡng (ví dụ 0.5 đối với bài toán nhị phân). Do không được tối ưu trực tiếp cho mục tiêu phân loại nên hiệu quả của Linear Regression thường thấp hơn Logistic Regression và phụ thuộc vào ngưỡng được lựa chọn.

### Kết luận

Đối với bài toán phân loại, đặc biệt là bài toán phân loại nhiều lớp như **Dry Bean Dataset**, **Logistic Regression** là lựa chọn phù hợp hơn **Linear Regression** vì:

- Được thiết kế chuyên biệt cho bài toán phân loại.
- Dự đoán xác suất và trực tiếp tối ưu ranh giới phân loại.
- Đạt độ chính xác, Precision, Recall và F1-score cao hơn.
- Không cần lựa chọn ngưỡng để chuyển đổi từ giá trị liên tục sang nhãn như Linear Regression.